# Fine-tune SmolVLA and ACT on Panthera-HT simulation data

Input: raw episodes from `tools/twin/twin_record.py`, attached as a Kaggle dataset. The dataset folder
contains `episode_*/`, `lerobot_convert.py` and `policy_bridge.py`.

Settings: GPU accelerator, Internet on (pretrained weights are downloaded).
`tools/train/kaggle_overnight.sh` uploads the data and starts this notebook unattended.

Both training runs write checkpoints every `SAVE_FREQ` steps, so a run cut short by the session
limit still leaves usable checkpoints. Each run logs to `/kaggle/working/<run>.log`.

In [ ]:
# Parameters
SMOLVLA_STEPS = 15000
ACT_STEPS = 20000
SAVE_FREQ = 2500
SMOLVLA_BATCH = 16
ACT_BATCH = 8

In [ ]:
# 1) Install LeRobot and check the GPU
!pip install -q "lerobot[smolvla]==0.4.4"
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2) Locate the raw dataset
import glob
import json
from collections import Counter
from pathlib import Path

candidates = [Path(p).parent for p in glob.glob("/kaggle/input/**/lerobot_convert.py", recursive=True)]
assert candidates, "attach the raw dataset (folder with lerobot_convert.py)"
RAW = candidates[0]
DATA = Path("/kaggle/working/panthera_twin")
REPO = "local/panthera_twin"
metas = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(RAW.glob("episode_*/meta.json"))]
print("raw folder:", RAW)
print("episodes:", len(metas), "successful:", sum(m.get("success", False) for m in metas))
print("languages:", Counter(m.get("language", "en") for m in metas))
print("levels:", Counter(m.get("level") for m in metas))

In [ ]:
# 3) Convert to a LeRobot dataset
!rm -rf {DATA}
!python {RAW}/lerobot_convert.py --raw {RAW} --root {DATA} --repo-id {REPO} 2>&1 | tail -n 20

## SmolVLA (language-conditioned)

`smolvla_base` expects cameras `camera1` to `camera3`: wrist maps to `camera1`, workspace to `camera2`,
`camera3` stays empty.

In [ ]:
RENAME = json.dumps({"observation.images.wrist": "observation.images.camera1",
                     "observation.images.workspace": "observation.images.camera2"})
SMOLVLA_OUT = "/kaggle/working/train_smolvla"
!lerobot-train --dataset.repo_id={REPO} --dataset.root={DATA} \
  --policy.path=lerobot/smolvla_base --policy.device=cuda --policy.push_to_hub=false \
  --policy.empty_cameras=1 --rename_map='{RENAME}' \
  --batch_size={SMOLVLA_BATCH} --steps={SMOLVLA_STEPS} --save_freq={SAVE_FREQ} --log_freq=100 \
  --num_workers=2 --output_dir={SMOLVLA_OUT} --job_name=panthera_twin_smolvla \
  --wandb.enable=false 2>&1 | tee /kaggle/working/smolvla.log | grep -E "step:|Error|error" | tail -n 40

## ACT baseline (no language input)

In [ ]:
ACT_OUT = "/kaggle/working/train_act"
!lerobot-train --dataset.repo_id={REPO} --dataset.root={DATA} \
  --policy.type=act --policy.device=cuda --policy.push_to_hub=false \
  --batch_size={ACT_BATCH} --steps={ACT_STEPS} --save_freq={SAVE_FREQ} --log_freq=100 \
  --num_workers=2 --output_dir={ACT_OUT} --job_name=panthera_twin_act \
  --wandb.enable=false 2>&1 | tee /kaggle/working/act.log | grep -E "step:|Error|error" | tail -n 40

## Package the last checkpoints (Output tab)

In [ ]:
!cd /kaggle/working && for d in train_smolvla train_act; do \
  [ -d $d/checkpoints/last/pretrained_model ] && tar czf ${d}_last.tgz -C $d/checkpoints/last pretrained_model; \
  done; ls -lh /kaggle/working/*.tgz /kaggle/working/*.log

## Run a checkpoint in the simulator (Ubuntu workstation)

1. LeRobot environment: `python tools/twin/policy_bridge.py serve --checkpoint <unpacked>/pretrained_model`
   (add `--camera-map wrist=camera1,workspace=camera2` for SmolVLA).
2. RoboTwin environment: `python tools/twin/twin_console.py --executor policy --policy-server 127.0.0.1:5555 --level 2`